In [1]:
import sys; sys.path.append("src")
from retrieval import rewrite_query, expand_query

q = "nyolong motor tetangga itu pasal apa"
r = rewrite_query(q, variant="v0_baseline")
print("rewrite :", r)
print("expand  :", expand_query(r, variant="v0_baseline"))

rewrite : Apa dasar hukum yang mengatur mengenai tindak pidana pencurian sepeda motor milik tetangga?
expand  : Apa dasar hukum yang mengatur mengenai tindak pidana pencurian sepeda motor milik tetangga? Pencurian kendaraan bermotor, Pengambilan barang milik orang lain secara melawan hukum


In [ ]:
import sys; sys.path.append("src")
from retrieval_langchain import (load_resources, extract_pasal_number,
                                 rewrite_query, expand_query)
from eval_query_prompts import DEV_QUERIES, transform_all, cetak_transformasi

# --- 1. Routing: memuat dokumen + index; model embedding belum di-load (lazy)
res = load_resources(index_path="data/index/baseline/faiss.index")

for q in ["isi pasal 476 itu apa", "pasal 9999 bunyinya apa", "nyolong motor tetangga itu pasal apa"]:
    nomor = extract_pasal_number(q)
    print(f"{q!r} ->", f"langsung ke pasal {nomor}" if nomor in res.pasal_index else "lanjut ke retrieval")

# --- 2. Rewrite + expansion: v0 (prompt lama) vs v1 (few-shot), lewat LangChain
T = transform_all(DEV_QUERIES[:3], rewrite_query, expand_query,
                  variants=["v0_baseline", "v1_fewshot"],
                  cache_path="data/eval/coba_langchain.json", tag="gemma4:12b-mlx-langchain")
cetak_transformasi(T)

'isi pasal 476 itu apa' -> langsung ke pasal 476
'pasal 9999 bunyinya apa' -> lanjut ke retrieval
'nyolong motor tetangga itu pasal apa' -> lanjut ke retrieval
[v0_baseline] 1/3: nyolong motor tetangga itu pasal apa


In [ ]:
import sys; sys.path.append("src")
from retrieval_langchain import rewrite_query, expand_query

for q in ["nyolong motor tetangga itu pasal apa",
          "ngerampas hp orang pakai todongan pisau"]:
    r = rewrite_query(q, variant="v1_fewshot")
    e = expand_query(r, variant="v1_fewshot")
    print(q, "\n  rewrite:", r, "\n  expand :", e, "\n")